In [28]:
import os
from io import BytesIO
import requests
from datetime import datetime,timedelta
import warnings
import xarray as xr
import io
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon, MultiPolygon
import pandas as pd
import matplotlib.patheffects as path_effects
from matplotlib.colors import ListedColormap
from matplotlib.colorbar import ColorbarBase
import matplotlib.colors as mcolors
#import image_functions as imagef
#import plotting_functions as plotf
import helper_dicts as hdict
import helper_functions as helper
%load_ext autoreload
%autoreload 2
import cftime

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
figure_dir = '/cpc/int_desk/pac_isl/stations/images'
gefs_dir = '/cpc/africawrf/ebekele/cca/xcast/subseason/prep_data/data/'

In [30]:
current_year = datetime.now().year
sst_daily = xr.open_mfdataset('/cpc/int_desk/data/oisstv2/sst.day.mean.*.nc', 
                              combine='by_coords')

In [31]:
climyear1 = 1991
climyear2 = 2020

#get a list of year strings from a start and an end date
def getYears(start, end):
    #start year, to end year inclusive
    # >get YearStrings ("1981", "1982"... "2016")
    #return [expression for var is iterable if condition]
    return [year for year in range(start, end+1)]
clim_years = getYears(climyear1, climyear2)
sst_clim = sst_daily.sel(time=sst_daily.time.dt.year.isin(clim_years))
sst_clim_mean = sst_clim.groupby('time.dayofyear').mean(dim = 'time')
# Step 1: Compute the day of year for each time in the original dataset
sst_daily_dayofyear = sst_daily.time.dt.dayofyear

# Step 2: Select matching climatology values for each day of the year
sst_climatology_expanded = sst_clim_mean.sel(dayofyear=sst_daily_dayofyear)

# Step 3: Calculate the anomaly by subtracting the climatology from the original dataset
sst_anomaly = sst_daily - sst_climatology_expanded
sst_anom_7 = sst_anomaly.isel(time=slice(-7,None)).mean(dim='time')
sst_anom_814 = sst_anomaly.isel(time=slice(-14,-7)).mean(dim='time')
sst_anom_diff = sst_anom_7 - sst_anom_814

/cpc/home/kkowal/.conda/envs/map_env/lib/python3.9/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/cpc/home/kkowal/.conda/envs/map_env/lib/python3.9/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing with an out-of-order index is generating 44 times more chunks
  return self.array[key]


In [32]:
import rioxarray

In [33]:
sst_anom_7 = sst_anom_7.rio.write_crs('EPSG:4326')
sst_anom7_mercator = sst_anom_7.rio.reproject('EPSG:3857')

sst_anom_diff = sst_anom_diff.rio.write_crs('EPSG:4326')
sst_anom7diff_mercator = sst_anom_diff.rio.reproject('EPSG:3857')

# Normalize the data to the desired range [-2, 2] (for example, SST anomalies)
vmin, vmax = -2, 2

In [38]:
sst_7norm

<xarray.DataArray 'sst' (y: 1605, x: 133)>
array([[ 2.,  2.,  2., ...,  2.,  2.,  2.],
       [ 2.,  2.,  2., ...,  2.,  2.,  2.],
       [ 2.,  2.,  2., ...,  2.,  2.,  2.],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], dtype=float32)
Coordinates:
  * x            (x) float64 -1.989e+07 -1.959e+07 ... 1.942e+07 1.972e+07
  * y            (y) float64 2.424e+08 2.421e+08 ... -2.422e+08 -2.425e+08
    spatial_ref  int64 0

In [34]:
# Clip values outside of the desired range (optional, but useful for extreme values)
sst_7norm = np.clip(sst_anom7_mercator.sst, vmin, vmax)
sst_7diffnorm = np.clip(sst_anom7diff_mercator.sst, vmin, vmax)

# Normalize the data so that it fits between 0 and 1
sst_7normalized = (sst_7norm - vmin) / (vmax - vmin)
sst_7diffnormalized = (sst_7diffnorm - vmin) / (vmax - vmin)

# Apply a color map (e.g., "RdBu" for anomalies)
cmap = plt.get_cmap("RdBu")

# Apply the color map to the normalized data
sst_7rgb = cmap(sst_7normalized)[:, :, :3]  # Extract RGB channels (ignore alpha channel)
sst_7rgb = (sst_7rgb * 255).astype(np.uint8)  # Scale to 0-255 for image representation

sst_7diffrgb = cmap(sst_7diffnormalized)[:, :, :3]  # Extract RGB channels (ignore alpha channel)
sst_7diffrgb = (sst_7diffrgb * 255).astype(np.uint8)  # Scale to 0-255 for image representation

# Convert to xarray for export
sst_7rgb_xarray = xr.DataArray(sst_7rgb, dims=("y", "x", "band"), 
                               coords={"y": sst_anom7_mercator.y, "x": sst_anom7_mercator.x, "band": [1, 2, 3]})
sst_7rgb_xarray = sst_7rgb_xarray.transpose("band", "y", "x")
sst_7rgb_xarray = sst_7rgb_xarray.rio.write_crs(sst_anom7_mercator.rio.crs)

sst_7diffrgb_xarray = xr.DataArray(sst_7diffrgb, dims=("y", "x", "band"),
                                   coords={"y": sst_anom7diff_mercator.y, "x": sst_anom7diff_mercator.x, "band": [1, 2, 3]})
sst_7diffrgb_xarray = sst_7diffrgb_xarray.transpose("band", "y", "x")
sst_7diffrgb_xarray = sst_7diffrgb_xarray.rio.write_crs(sst_anom7diff_mercator.rio.crs)

#check crs is right
sst_7rgb_xarray = sst_7rgb_xarray.rio.write_crs("EPSG:3857")
sst_7diffrgb_xarray = sst_7diffrgb_xarray.rio.write_crs("EPSG:3857")
# Export to GeoTIFF
sst_7rgb_xarray.rio.to_raster(os.path.join(figure_dir, 'sst_mercator7.tif'))
sst_7diffrgb_xarray.rio.to_raster(os.path.join(figure_dir, 'sst_mercator7diff.tif'))


In [ ]:
#command line
# gdal2tiles.py -p mercator -z 0-5 /cpc/int_desk/pac_isl/stations/images/sst_mercator7.tif /cpc/int_desk/pac_isl/stations/images/sst_anom7_tiles
# gdal2tiles.py -p mercator -z 0-5 /cpc/int_desk/pac_isl/stations/images/sst_mercator7diff.tif /cpc/int_desk/pac_isl/stations/images/sst_anom7diff_tiles

In [8]:
gefs_week1 = xr.open_dataset(os.path.join(gefs_dir, 'gefs_week1_fcst.nc'))

<xarray.Dataset>
Dimensions:  (lat: 201, lon: 721, time: 1)
Coordinates:
  * lat      (lat) float32 -50.0 -49.5 -49.0 -48.5 -48.0 ... 48.5 49.0 49.5 50.0
  * lon      (lon) float32 -180.0 -179.5 -179.0 -178.5 ... 179.0 179.5 180.0
  * time     (time) datetime64[ns] 2024-11-08
Data variables:
    precip   (time, lat, lon) float64 ...